In [1]:
# QTemp 網頁版。將本檔完整貼入同一個 Google Colab 程式碼儲存格並執行。
import sys
import subprocess
from importlib.util import find_spec

dependencies = {"yfinance": "yfinance", "exchange_calendars": "exchange-calendars"}
missing = [package for module, package in dependencies.items() if find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

from collections import deque
import numpy as np
import pandas as pd

SIGNALS = {'buy': '低估值', 'sell': '高估值', 'bull': '底背離', 'bear': '頂背離'}
COLORS = {'buy': '#72a7ff', 'sell': '#ff8a88', 'bull': '#7ce7ba', 'bear': '#ffd08b'}

def ema(previous, value, length):
    alpha = 2 / (length + 1)
    return value if previous is None else alpha * value + (1 - alpha) * previous

def wilder(previous, value, seed):
    if previous is not None:
        alpha = 1 / 14
        return alpha * value + (1 - alpha) * previous
    seed.append(value)
    return sum(seed) / 14 if len(seed) == 14 else None

def midrank(series, index, length, minimum):
    value = series[index]
    if not np.isfinite(value):
        return None, 0
    prior = series[max(0, index - length):index]
    valid = prior[np.isfinite(prior)]
    count = len(valid)
    if count < minimum or count < len(prior) * .6:
        return None, count
    # Pine comparison operators round each float operand to nine decimals.
    valid = np.round(valid, 9)
    low, high = np.round([value - 1e-10, value + 1e-10], 9)
    below = np.count_nonzero(valid < low)
    tied = np.count_nonzero((valid >= low) & (valid <= high))
    return (below + .5 * tied) / count, count

def path_clear(opens, closes, scores, atr, a, b, bull, tick):
    ai, ap, ac = a
    bi, bp, bc = b
    distance = bi - ai
    cs, ps = (bc-ac)/distance, (bp-ap)/distance
    hits = touches = 0
    last = -1.
    lo, hi = min(ac, bc), max(ac, bc)
    for step in range(distance):
        index = ai + step
        current, following = scores[index:index+2]
        lo, hi = min(lo, current, following), max(hi, current, following)
        if round(hi - lo,9) > round(120 + 1e-8,9):
            return False
        contacts = []
        if round(abs(cs),9) > 0:
            x = (current-ac)/cs
            if step - 1e-7 <= x <= step + 1 + 1e-7:
                contacts.append(x)
        elif round(abs(current - (ac + cs * step)),9) <= 1e-8:
            return False
        right = ac + cs * (step+1)
        if (step+1 < distance and abs(following-current) > 1e-8 and
                min(current, following)-1e-8 <= right <= max(current, following)+1e-8):
            contacts.append(step + 1)
        for x in contacts:
            if 1e-7 < x < distance - 1e-7 and abs(x-last) > 1e-7:
                hits, last = hits+1, x
        if hits > 3:
            return False
        if step:
            o, c = opens[index], closes[index]
            left, right_price = ap+ps*(step-.5), ap+ps*(step+.5)
            scale = max(atr[index] if np.isfinite(atr[index]) else abs(c-o), tick)
            penetration = (max(left, right_price)-min(o, c) if bull else max(o, c)-min(left, right_price))/scale
            if penetration > max(.05, tick/scale) + 1e-8:
                return False
            touches += penetration >= -1e-8
            if touches > 1:
                return False
    return True

def qtemp(frame, eps=None, tick=.01):
    o, h, l, c, v = (frame[col].to_numpy(dtype=float) for col in ['Open','High','Low','Close','Volume'])
    size = len(c)
    divs, pe_history, smooth, atr, scores = [np.full(size, np.nan) for _ in range(5)]
    rsi_out, pe_weights, warmup = np.full(size,np.nan), np.zeros(size), np.zeros(size,dtype=bool)
    fast = slow = gain = loss = flow_ema = atr_value = blend = None
    cumulative = 0.
    seeds = [[],[],[]]
    highs, lows = deque(maxlen=100), deque(maxlen=100)
    events = []
    eps_values = eps.reindex(frame.index).to_numpy(dtype=float) if eps is not None else np.full(size, np.nan)
    pe_start = max(0, size-5000)
    pe_window = pe_history[pe_start:]
    for i in range(size):
        if i:
            delta = c[i]-c[i-1]
            gain = wilder(gain, max(delta,0), seeds[0])
            loss = wilder(loss, max(-delta,0), seeds[1])
        rsi = np.nan if gain is None or loss is None else 100 if loss == 0 else 100-100/(1+gain/loss)
        rsi_out[i] = rsi
        cci_value = np.nan
        if i>=19:
            window = c[i-19:i+1]
            mean = window.mean()
            mad = np.abs(window-mean).mean()
            if mad:
                cci_value = (c[i]-mean)/(.015*mad)
        votes = int(np.isfinite(rsi))+int(np.isfinite(cci_value))
        rsi_compare, cci_compare = round(rsi,9), round(cci_value,9)
        momentum = (int(rsi_compare>60)-int(rsi_compare<40)+int(cci_compare>100)-int(cci_compare<-100))/votes if votes else 0
        fast, slow = ema(fast,c[i],50), ema(slow,c[i],200)
        old_flow = flow_ema
        candle_side = np.sign(round(c[i],9)-round(o[i],9))
        cumulative += candle_side * (v[i] if np.isfinite(v[i]) else 0)
        flow_ema = ema(flow_ema,cumulative,21)
        vw = 20 if np.isfinite(v[i]) and old_flow is not None and np.any(v[max(0,i-20):i+1]>0) else 0
        slope = np.sign(round(flow_ema-old_flow,9)) if old_flow is not None else 0
        consensus = (45*np.sign(round(fast,9)-round(slow,9))+35*(int(momentum>.25)-int(momentum<-.25))+vw*slope)/(45+35*bool(votes)+vw)
        divs[i] = np.log(c[i]/slow)
        deviation, _ = midrank(divs,i,504,126)
        if i >= pe_start and np.isfinite(eps_values[i]) and round(eps_values[i],9)>0:
            pe_history[i] = np.log(c[i])-np.log(eps_values[i])
        pe, observations = midrank(pe_window,i-1-pe_start,756,1) if i>pe_start else (None,0)
        pe_weight = 0 if pe is None else 15*min(1,max(0,(observations-126)/126))
        total = 35+(50 if deviation is not None else 0)+pe_weight
        value = ((35/total)*consensus+((50/total)*(2*deviation-1) if deviation is not None else 0)+
                 ((pe_weight/total)*(2*pe-1) if pe is not None else 0))
        blend = ema(blend,value,3)
        smooth[i] = blend
        percentile, count = midrank(smooth,i,504,1)
        scores[i] = np.clip(100*(percentile-.5)/.45,-100,100) if percentile is not None and count>=30 else 100*blend
        warmup[i], pe_weights[i] = count<126, pe_weight/total
        true_range = max(h[i]-l[i],abs(h[i]-c[i-1]),abs(l[i]-c[i-1])) if i else h[i]-l[i]
        atr_value = wilder(atr_value,true_range,seeds[2])
        if atr_value is not None:
            atr[i] = atr_value
        if rsi_compare < 30 and round(scores[i],9) < -90:
            events.append(('buy',i,i,None))
        if rsi_compare > 70 and round(scores[i],9) > 90:
            events.append(('sell',i,i,None))
        if i<6:
            continue
        pivot = i-3
        for bull, prices, side, anchors in [(False,h,1,highs),(True,l,-1,lows)]:
            price = prices[pivot]
            if any((prices[j]-price)*side > 0 or (j>pivot and prices[j]==price) for j in range(i-6,i+1)):
                continue
            while anchors and pivot-anchors[0][0]>90:
                anchors.popleft()
            score = scores[pivot]
            current = pivot,price,score
            if np.sign(round(c[pivot],9)-round(o[pivot],9))==side and (round(score,9)<=50 if bull else round(score,9)>=-50):
                for anchor in anchors:
                    j, start_price, start_score = anchor
                    if (pivot-j>=12 and np.sign(round(c[j],9)-round(o[j],9))==side
                            and (round(price,9)-round(start_price,9))*side>0
                            and (round(score,9)-round(start_score,9))*side<0
                            and path_clear(o,c,scores,atr,anchor,current,bull,tick)):
                        events.append(('bull' if bull else 'bear',pivot,i,j))
            anchors.append(current)
    values = pd.DataFrame({'QTemp':scores,'RSI':rsi_out,'P/E 權重':pe_weights,'暫算':warmup},index=frame.index)
    rows = [{'類型':SIGNALS[k],'代碼':k,'標記日':frame.index[j],'確認日':frame.index[i],
             '確認收盤價':c[i],'暫算':bool(warmup[i]),'標記位置':j,'確認位置':i,'背離起點':a}
            for k,j,i,a in events]
    return values, pd.DataFrame(rows,columns=['類型','代碼','標記日','確認日','確認收盤價','暫算','標記位置','確認位置','背離起點'])

from functools import lru_cache
from datetime import datetime, time
import re
import unicodedata
from zoneinfo import ZoneInfo
import yfinance as yf

def symbol_input(value):
    value = unicodedata.normalize('NFKC', value).strip().upper()
    return value.replace(' ', '') if re.fullmatch(r'[A-Z0-9.^=_ -]+', value) else value

def candidates(value):
    ticker = symbol_input(value)
    if re.fullmatch(r'\d{4,6}[A-Z]?',ticker):
        return [ticker+'.TW', ticker+'.TWO']
    return [ticker]

def search_yahoo(text):
    text = unicodedata.normalize('NFKC', text).strip()
    if not text:
        return []
    results = list(dict.fromkeys(candidates(text))) if re.fullmatch(r'[A-Z0-9.^=_ -]+', text.upper()) else []
    choices = [(s,s) for s in results]
    try:
        quotes = yf.Search(text, max_results=12, news_count=0, enable_fuzzy_query=False).quotes
        for item in quotes:
            symbol = item.get('symbol')
            if symbol and item.get('quoteType') in {'EQUITY','ETF','INDEX','MUTUALFUND'}:
                label = f"{symbol} ｜ {item.get('shortname') or item.get('longname') or ''} ｜ {item.get('exchDisp') or ''}"
                choices = [(old_label,s) for old_label,s in choices if s != symbol]
                choices.append((label,symbol))
    except Exception:
        if not choices:
            raise
    return choices

def closed_cutoff(local_date, tz):
    try:
        import exchange_calendars as xcals
        names = {'Asia/Taipei': 'XTAI', 'America/New_York': 'XNYS'}
        calendar = xcals.get_calendar(names[tz])
        return calendar.session_close(pd.Timestamp(local_date)).to_pydatetime() + pd.Timedelta(minutes=20)
    except Exception:
        return datetime.combine(local_date, time(17,30), ZoneInfo(tz))

@lru_cache(maxsize=12)
def yahoo_history(symbol, refresh_slot):
    ticker = yf.Ticker(symbol)
    frame = ticker.history(period='max', interval='1d', auto_adjust=False, actions=True,
                           prepost=False, timeout=20, raise_errors=True)
    if frame.empty:
        raise ValueError(f'Yahoo Finance 未回傳 {symbol} 的歷史日線。')
    metadata = ticker.history_metadata or {}
    zone = str(getattr(frame.index, 'tz', '') or metadata.get('exchangeTimezoneName') or '')
    if not zone or zone == 'UTC':
        zone = 'Asia/Taipei' if symbol.endswith(('.TW','.TWO')) else 'America/New_York'
    now = datetime.now(ZoneInfo(zone))
    dates = pd.DatetimeIndex(frame.index)
    if dates.tz is not None:
        dates = dates.tz_convert(zone).tz_localize(None)
    frame.index = dates.normalize().rename('交易日')
    frame = frame.loc[~frame.index.duplicated(keep='last')].sort_index()
    today = pd.Timestamp(now.date())
    frame = frame[frame.index <= today]
    if now < closed_cutoff(now.date(),zone):
        frame = frame[frame.index < today]
    columns = ['Open','High','Low','Close','Adj Close','Volume','Dividends','Stock Splits']
    frame = frame.reindex(columns=columns).apply(pd.to_numeric,errors='coerce')
    ohlc = frame[['Open','High','Low','Close']]
    valid = np.isfinite(ohlc).all(axis=1) & ohlc.gt(0).all(axis=1)
    valid &= frame.High.ge(ohlc[['Open','Low','Close']].max(axis=1))
    valid &= frame.Low.le(ohlc[['Open','High','Close']].min(axis=1))
    frame = frame.loc[valid].copy()
    frame['Volume'] = frame.Volume.where(np.isfinite(frame.Volume) & frame.Volume.ge(0))
    if len(frame)<20:
        raise ValueError(f'{symbol} 只有 {len(frame)} 根已收盤日 K，至少需要 20 根。')
    return frame,metadata,zone

def load_market(value):
    failures = []
    slot = int(pd.Timestamp.now(tz='UTC').timestamp() // 300)
    for symbol in candidates(value):
        try:
            frame, meta, zone = yahoo_history(symbol,slot)
            return symbol,frame.copy(),meta,zone
        except Exception as error:
            failures.append(f'{symbol}：{error}')
    raise RuntimeError('；'.join(failures))

def apply_price_basis(frame):
    factor = (frame['Adj Close']/frame.Close).replace([np.inf,-np.inf],np.nan)
    usable = factor.notna() & factor.gt(0)
    if not usable.all():
        raise ValueError('Yahoo Finance 的調整收盤價有缺漏，無法用 Adj Close 還原。')
    result = frame.copy()
    result[['Open','High','Low']] = frame[['Open','High','Low']].mul(factor,axis=0)
    result['Close'] = frame['Adj Close']
    result['High'] = result.High.clip(lower=result.Close)
    result['Low'] = result.Low.clip(upper=result.Close)
    return result

def optional_info(ticker):
    try:
        return ticker.get_info() or {},None
    except Exception as error:
        return {},str(error)

def annual_table(frame, events, years=10, event='低估值', include_warmup=True):
    closes = frame.Close.groupby(frame.index.year).last()
    latest_year = int(frame.index[-1].year)
    selected = events[events['類型'].eq(event)]
    if not include_warmup:
        selected = selected[~selected['暫算']]
    selected = selected.drop_duplicates(['類型','確認日'])
    event_years = pd.to_datetime(selected['確認日']).dt.year
    rows = []
    for year in closes.index[::-1]:
        if year<latest_year-years+1:
            break
        e = selected.loc[event_years.eq(year),'確認收盤價']
        previous = closes.get(year-1, np.nan)
        avg = e.mean()
        rows.append({'年度':int(year),'訊號次數':len(e),'訊號均價':avg,
                     '均價至最近收盤報酬率':frame.Close.iloc[-1]/avg-1 if len(e) else np.nan,
                     '年度報酬率':closes.loc[year]/previous-1})
    return pd.DataFrame(rows)

def historical_returns(frame):
    last = frame.index[-1]
    targets = {'1 個月':pd.DateOffset(months=1),'3 個月':pd.DateOffset(months=3),
               '1 年':pd.DateOffset(years=1),'3 年':pd.DateOffset(years=3),
               '5 年':pd.DateOffset(years=5),'10 年':pd.DateOffset(years=10)}
    rows = []
    for label, offset in targets.items():
        i = frame.index.searchsorted(last-offset,side='right')-1
        rows.append({'期間':label,'基準日':frame.index[i] if i>=0 else pd.NaT,
                     '報酬率':frame.Close.iloc[-1]/frame.Close.iloc[i]-1 if i>=0 else np.nan})
    return pd.DataFrame(rows)

import json
from uuid import uuid4
from IPython.display import HTML, JSON, display

from io import StringIO

def chart_payload(frame, values, events):
    first = frame.index.searchsorted(frame.index[-1] - pd.DateOffset(years=12))
    visible = frame.iloc[first:]
    days = visible.index.strftime('%Y-%m-%d').tolist()
    scores = values.QTemp.iloc[first:].to_list()
    candles = [[day, float(o), float(h), float(l), float(c), None if pd.isna(v) else float(v)]
               for day, (o,h,l,c,v) in zip(days, visible[['Open','High','Low','Close','Volume']].itertuples(index=False,name=None))]
    selected = events.loc[events['標記位置'].ge(first)]
    marked = selected.drop_duplicates(['標記位置','代碼']).sort_values('標記位置',kind='stable')
    marks = [[days[int(position)-first], code] for position, code in
             marked[['標記位置','代碼']].itertuples(index=False,name=None)]
    segments = []
    for end, code, anchor in selected[['標記位置','代碼','背離起點']].dropna().tail(150).itertuples(index=False,name=None):
        if anchor < first:
            continue
        a,b = int(anchor)-first,int(end)-first
        price_index = 3 if code == 'bull' else 2
        segments.append([days[a],days[b],candles[a][price_index],candles[b][price_index],scores[a],scores[b],code])
    return {'candles':candles,'scores':scores,'marks':marks,'segments':segments}

def tv_csv_frame(csv_text, zone):
    if len(csv_text)>2_000_000:
        raise ValueError('CSV 大於 2 MB，請縮短匯出區間。')
    raw = pd.read_csv(StringIO(csv_text.lstrip('\ufeff')))
    columns = {str(col).strip().lower():col for col in raw.columns}
    date_col = next((columns[k] for k in ('time','date','日期','時間') if k in columns),None)
    if date_col is None or not all(k in columns for k in ('open','high','low','close')):
        raise ValueError('CSV 須有 TradingView 的 Time、Open、High、Low、Close 欄。')
    times = raw[date_col].astype(str).str.strip()
    numeric = pd.to_numeric(times,errors='coerce')
    if numeric.notna().all():
        unit = 'ms' if numeric.abs().median()>1e11 else 's'
        index = pd.DatetimeIndex(pd.to_datetime(numeric,unit=unit,utc=True))
        if not (index == index.normalize()).all():
            index = index.tz_convert(zone)
        index = index.tz_localize(None)
    else:
        parsed = [pd.Timestamp(stamp) for stamp in times]
        index = pd.DatetimeIndex([stamp if stamp.tzinfo is None else stamp.tz_convert(zone).tz_localize(None)
                                  for stamp in parsed])
    if index.hasnans:
        raise ValueError('CSV 有無效的交易日。')
    frame = pd.DataFrame(index=index.normalize().rename('交易日'))
    if frame.index.has_duplicates:
        raise ValueError('CSV 有同日多筆資料，請用日 K 匯出。')
    for k in ('open','high','low','close','volume'):
        col = columns.get(k) or (columns.get('成交量') if k == 'volume' else None)
        frame[k.title()] = pd.to_numeric(raw[col],errors='coerce').to_numpy() if col is not None else np.nan
    target = next((col for key,col in columns.items() if 'relative temperature' in key or key in ('qtemp','qtemp score','qtemp 分數')),None)
    if target is None:
        raise ValueError('CSV 缺少 QTemp 分數，請把 QTemp 指標加到 TradingView 圖表後再匯出。')
    frame['TradingView'] = pd.to_numeric(raw[target],errors='coerce').to_numpy()
    ohlc = frame[['Open','High','Low','Close']]
    if not (np.isfinite(ohlc).all(axis=1) & ohlc.gt(0).all(axis=1) & frame.High.ge(ohlc.max(axis=1)) & frame.Low.le(ohlc.min(axis=1))).all():
        raise ValueError('CSV 有無效的日 K 價格。')
    frame['Volume'] = frame.Volume.where(np.isfinite(frame.Volume) & frame.Volume.ge(0))
    if not frame.TradingView.notna().any():
        raise ValueError('CSV 的 QTemp 欄沒有數值。')
    return frame.sort_index()

def site_compare(requested, csv_text):
    try:
        symbol, raw, meta, zone = load_market(requested)
        yahoo = apply_price_basis(raw)
        frame = tv_csv_frame(csv_text,zone)
        now = datetime.now(ZoneInfo(zone))
        today = pd.Timestamp(now.date())
        frame = frame.loc[frame.index < today if now < closed_cutoff(now.date(),zone) else frame.index <= today]
        if len(frame)<20:
            raise ValueError('CSV 至少需要 20 根已收盤日 K。')
        tick = 10**(-int(meta.get('priceHint') or 2))
        same, _ = qtemp(frame, tick=tick)
        # Align start dates when comparing providers so EMA warm-up is comparable.
        span = yahoo.loc[frame.index[0]:frame.index[-1]]
        if span.empty:
            raise ValueError('Yahoo 與 CSV 的交易日沒有重疊，請核對標的。')
        yvalues, _ = qtemp(span,tick=tick)
        audit = pd.DataFrame({'TV 分數':frame.TradingView,'同一 CSV 重算':same.QTemp,'Yahoo 重算':yvalues.QTemp,
                              'TV 收盤':frame.Close,'Yahoo 收盤':span.Close,
                              'TV 成交量':frame.Volume,'Yahoo 成交量':span.Volume}).reindex(frame.index)
        audit['CSV 重算差'] = audit['同一 CSV 重算'] - audit['TV 分數']
        audit['Yahoo 重算差'] = audit['Yahoo 重算'] - audit['TV 分數']
        valid = audit[['CSV 重算差','Yahoo 重算差']].dropna()
        def different(a,b):
            comparable = audit[[a,b]].dropna()
            return int((~np.isclose(comparable[a],comparable[b],rtol=1e-7,atol=1e-8)).sum()),len(comparable)
        price_count, price_days = different('TV 收盤','Yahoo 收盤')
        vol_count, vol_days = different('TV 成交量','Yahoo 成交量')
        ohlc = ['Open','High','Low','Close']
        common = frame.index.intersection(span.index)
        different_ohlc = (~np.isclose(frame.loc[common,ohlc],span.loc[common,ohlc],rtol=1e-7,atol=1e-8)).any(axis=1).sum()
        volume_note = f'成交量不同 {vol_count}/{vol_days} 日' if vol_days else '無共同成交量可比對'
        if not frame.Volume.notna().any():
            volume_note = 'CSV 缺少有效成交量，同一 CSV 重算的量能權重為零'
        warm_note = '｜CSV 歷史不足 1,200 根，分位與 EMA 仍可能受圖表起點影響' if len(frame)<1200 else ''
        score_note = f'｜共同分數 {len(valid)} 日，平均絕對差：同一 CSV {valid["CSV 重算差"].abs().mean():.4f}、Yahoo {valid["Yahoo 重算差"].abs().mean():.4f}' if len(valid) else ''
        return JSON({'table':records(audit.tail(50).iloc[::-1].reset_index()),
                     'note': f'{symbol}｜OHLC 不同 {different_ohlc}/{len(common)} 日，其中收盤不同 {price_count}/{price_days} 日｜{volume_note}{score_note}{warm_note}。'})
    except Exception as error:
        return JSON({'error':f'無法對照：{error}'})

def market_details(symbol, frame, info, meta, zone):
    price = info.get('regularMarketPrice')
    stamp = info.get('regularMarketTime')
    try:
        quote_time = pd.to_datetime(stamp,unit='s',utc=True).tz_convert(zone).strftime('%Y-%m-%d %H:%M:%S %Z') if stamp else None
    except (TypeError, ValueError, OverflowError):
        quote_time = None
    fields = [('名稱',info.get('longName') or info.get('shortName')),('代號',symbol),
              ('交易所',info.get('fullExchangeName') or meta.get('fullExchangeName') or meta.get('exchangeName')),
              ('幣別',info.get('currency') or meta.get('currency')),('交易所時區',zone),
              ('Yahoo 報價',price),('Yahoo 報價時間',quote_time),('Yahoo 市況',info.get('marketState')),
              ('Yahoo 報價成交量',info.get('regularMarketVolume')),('Yahoo 漲跌幅',info.get('regularMarketChangePercent')),
              ('最近日 K 日期',frame.index[-1].date().isoformat()),('最近日 K 還原收盤',frame.Close.iloc[-1]),
              ('最近日 K 成交量',frame.Volume.iloc[-1]),('市值',info.get('marketCap')),
              ('近四季 EPS',info.get('trailingEps')),('本益比',info.get('trailingPE')),
              ('預估本益比',info.get('forwardPE')),('股價淨值比',info.get('priceToBook')),
              ('52 週最高',info.get('fiftyTwoWeekHigh')),('52 週最低',info.get('fiftyTwoWeekLow'))]
    return pd.DataFrame([{'項目':k,'數值':f'{v:,.4f}'.rstrip('0').rstrip('.') if isinstance(v,(float,np.floating)) and np.isfinite(v)
                          else f'{v:,}' if isinstance(v,(int,np.integer)) else str(v) if v is not None else '—'} for k,v in fields])

def records(frame):
    return {
        "columns": [str(c) for c in frame.columns],
        "rows": json.loads(frame.to_json(orient='values', date_format='iso', double_precision=10)),
    }

def site_search(query):
    try:
        return JSON({'choices': search_yahoo(str(query)[:100])})
    except Exception as error:
        return JSON({'error': f'Yahoo Finance 搜尋失敗：{error}'})

def site_analyze(requested):
    try:
        requested = symbol_input(str(requested)[:100])
        if not re.fullmatch(r'[A-Z0-9.^=_-]{1,30}', requested):
            raise ValueError('公司名稱請從搜尋結果選取。')
        symbol, raw, meta, zone = load_market(requested)
        frame = apply_price_basis(raw)
        ticker = yf.Ticker(symbol)
        info, info_error = optional_info(ticker)
        tick = 10**(-int(meta.get('priceHint') or 2))
        values, events = qtemp(frame, tick=tick)
        figure = chart_payload(frame, values, events)
        newest = values.iloc[-1]
        annual = {name: records(annual_table(frame,events,10,name)) for name in SIGNALS.values()}
        event_rows = events[events['確認日'].ge(pd.Timestamp(frame.index[-1].year-9,1,1))].drop_duplicates(['類型','確認日'])
        recent = frame.drop(columns='Adj Close').join(values[['QTemp','RSI']]).tail(120).reset_index()
        actions = raw.loc[raw.Dividends.fillna(0).ne(0) | raw['Stock Splits'].fillna(0).ne(0),['Dividends','Stock Splits']]
        return JSON({
            'symbol': symbol, 'name': info.get('shortName') or info.get('longName') or symbol,
            'market': info.get('fullExchangeName') or meta.get('fullExchangeName') or meta.get('exchangeName'),
            'currency': info.get('currency') or meta.get('currency'), 'zone': zone,
            'start': frame.index[0].date().isoformat(), 'end': frame.index[-1].date().isoformat(),
            'last': float(frame.Close.iloc[-1]), 'change': float(frame.Close.iloc[-1] / frame.Close.iloc[-2] - 1),
            'volume': None if pd.isna(raw.Volume.iloc[-1]) else float(raw.Volume.iloc[-1]),
            'qtemp': float(newest.QTemp), 'basis': 'Adj Close 還原', 'infoMissing': bool(info_error),
            'figure': figure,
            'annual': annual, 'returns': records(historical_returns(frame)),
            'events': records(event_rows[['類型','標記日','確認日','確認收盤價']].iloc[::-1]),
            'details': records(market_details(symbol,frame,info,meta,zone)),
            'daily': records(recent.iloc[::-1]), 'actions': records(actions.tail(50).reset_index().iloc[::-1]),
        })
    except Exception as error:
        return JSON({'error': f'無法載入：{error}'})

def display_site():
    from google.colab import output
    root = 'qtemp_' + uuid4().hex
    output.register_callback(root + '.search', site_search)
    output.register_callback(root + '.analyze', site_analyze)
    output.register_callback(root + '.compare', site_compare)
    display(HTML(PAGE.replace('__QTEMP_ROOT__', root)))

PAGE = r"""
<style>
#__QTEMP_ROOT__{
  --ink:#142238;--muted:#526176;--line:#dce4ed;--surface:#f5f8fc;
  --navy:#203b60;--gold:#895c18;--up:#b6293b;--down:#117454;
  background:#fff;color:var(--ink);color-scheme:light;box-sizing:border-box;
  width:100%;max-width:1320px;margin:0 auto;padding:clamp(16px,3vw,36px);
  font:14px/1.5 Inter,"Noto Sans TC",system-ui,sans-serif;
}
#__QTEMP_ROOT__ *{box-sizing:border-box}
#__QTEMP_ROOT__ h1,#__QTEMP_ROOT__ h2,#__QTEMP_ROOT__ h3{margin:0;line-height:1.25;font-weight:700}
#__QTEMP_ROOT__ h1{font-size:clamp(23px,3.2vw,39px);letter-spacing:-.035em}
#__QTEMP_ROOT__ h2{font-size:clamp(20px,2.4vw,27px);letter-spacing:-.035em;overflow-wrap:anywhere}
#__QTEMP_ROOT__ h3{font-size:15px;margin:22px 0 10px}
#__QTEMP_ROOT__ .muted{color:var(--muted)}
#__QTEMP_ROOT__ .head,#__QTEMP_ROOT__ .controls,#__QTEMP_ROOT__ .filters,#__QTEMP_ROOT__ .tabs{
  display:flex;align-items:center;gap:8px;flex-wrap:wrap
}
#__QTEMP_ROOT__ .head{justify-content:space-between;gap:12px;margin-bottom:19px}
#__QTEMP_ROOT__ .source{display:flex;align-items:center;gap:7px;font-size:12px;font-weight:600}
#__QTEMP_ROOT__ .source::before{content:"";width:7px;height:7px;border-radius:50%;background:#6e819d}
#__QTEMP_ROOT__ .query{padding:14px;border:1px solid var(--line);border-radius:14px;background:var(--surface)}
#__QTEMP_ROOT__ .controls input[type=search]{flex:1;min-width:140px}
#__QTEMP_ROOT__ input::placeholder{color:#627086;opacity:1}
#__QTEMP_ROOT__ button,#__QTEMP_ROOT__ input,#__QTEMP_ROOT__ select{
  font:inherit;color:var(--ink);background:#fff;border:1px solid #cdd8e5;border-radius:8px;padding:8px 11px
}
#__QTEMP_ROOT__ button{font-weight:600;cursor:pointer}
#__QTEMP_ROOT__ button:hover:not(:disabled){background:#edf3fa;border-color:#a8bbd1}
#__QTEMP_ROOT__ button:disabled{opacity:.55;cursor:wait}
#__QTEMP_ROOT__ :is(button,input,select):focus-visible{outline:2px solid #406b9b;outline-offset:2px}
#__QTEMP_ROOT__ .primary{background:var(--navy);color:#fff;border-color:var(--navy)}
#__QTEMP_ROOT__ .primary:hover:not(:disabled){background:#2b4b76;border-color:#2b4b76}
#__QTEMP_ROOT__ .filters label{display:inline-flex;align-items:center;gap:6px;max-width:100%}
#__QTEMP_ROOT__ input[type=file],#__QTEMP_ROOT__ select[data-q=matches]{max-width:100%}
#__QTEMP_ROOT__ [data-q=comparetools] label{flex-wrap:wrap}
#__QTEMP_ROOT__ input[type=file]{min-width:0}
#__QTEMP_ROOT__ .status{min-height:20px;margin:8px 1px 0;font-size:12px}
#__QTEMP_ROOT__ .status.error{color:var(--up);font-weight:600}
#__QTEMP_ROOT__ .asset{display:flex;align-items:baseline;gap:5px 13px;flex-wrap:wrap;margin:22px 0 12px}
#__QTEMP_ROOT__ .asset h2{color:#0a1424;font-weight:750}
#__QTEMP_ROOT__ .asset .muted{font-size:12px;font-variant-numeric:tabular-nums}
#__QTEMP_ROOT__ .tiles{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:10px}
#__QTEMP_ROOT__ .tile{
  background:#fff;border:1px solid var(--line);border-radius:12px;padding:13px 15px;
  box-shadow:0 2px 5px #213b5c08;color:var(--muted);font-size:12px;font-weight:600
}
#__QTEMP_ROOT__ .tile strong{
  display:block;margin-top:4px;color:var(--ink);font-size:clamp(17px,2.45vw,28px);line-height:1.25;
  font-variant-numeric:tabular-nums;font-weight:700;letter-spacing:-.045em;overflow-wrap:anywhere
}
#__QTEMP_ROOT__ .tile.qtemp{background:#fdf8ef;border-color:#e9d9bc;color:var(--gold)}
#__QTEMP_ROOT__ .tile.qtemp strong{color:var(--gold)}
#__QTEMP_ROOT__ .tile strong.positive{color:var(--up)}
#__QTEMP_ROOT__ .tile strong.negative{color:var(--down)}
#__QTEMP_ROOT__ .tabs{flex-wrap:nowrap;overflow-x:auto;gap:13px;margin:15px 0 12px;border-bottom:1px solid var(--line)}
#__QTEMP_ROOT__ .tabs button{
  flex:none;white-space:nowrap;padding:11px 5px 10px;border:0;border-bottom:2px solid transparent;
  border-radius:0;color:var(--muted)
}
#__QTEMP_ROOT__ .tabs button:hover{background:var(--surface)}
#__QTEMP_ROOT__ .tabs button:focus-visible{outline-offset:-3px}
#__QTEMP_ROOT__ .tabs button[aria-pressed=true]{color:var(--navy);border-bottom-color:var(--navy)}
#__QTEMP_ROOT__ .panel{
  min-width:0;padding:clamp(10px,1.6vw,18px);background:#fff;border:1px solid var(--line);
  border-radius:14px;box-shadow:0 4px 16px #213b5c0a
}
#__QTEMP_ROOT__ .chart-tools{justify-content:space-between;margin-bottom:8px}
#__QTEMP_ROOT__ .chart-tools button{padding:6px 10px;font-size:12px}
#__QTEMP_ROOT__ .range{display:flex;flex-wrap:wrap;gap:5px}
#__QTEMP_ROOT__ .chart-tools [data-q=reset-chart]{color:var(--muted)}
#__QTEMP_ROOT__ .ohlc{
  display:flex;align-items:baseline;gap:4px 13px;flex-wrap:wrap;
  margin:0 0 8px;padding:7px 10px;background:var(--surface);border-radius:8px;
  font-size:12px;font-variant-numeric:tabular-nums
}
#__QTEMP_ROOT__ .ohlc span{white-space:nowrap;color:var(--muted)}
#__QTEMP_ROOT__ .ohlc strong{color:var(--ink);margin-left:3px;font-weight:650}
#__QTEMP_ROOT__ .ohlc span:last-child strong{color:var(--gold)}
#__QTEMP_ROOT__ .plot{height:660px;min-width:0}
#__QTEMP_ROOT__ .table{overflow:auto;max-height:580px;border-radius:8px;margin-top:7px}
#__QTEMP_ROOT__ table{border-collapse:separate;border-spacing:0;width:100%;font-variant-numeric:tabular-nums}
#__QTEMP_ROOT__ th,#__QTEMP_ROOT__ td{text-align:right;padding:10px 12px;white-space:nowrap;border-bottom:1px solid #e7edf4}
#__QTEMP_ROOT__ th:first-child,#__QTEMP_ROOT__ td:first-child{text-align:left}
#__QTEMP_ROOT__ th{position:sticky;top:0;background:#edf3fa;color:#3f536d;font-size:12px;font-weight:700}
#__QTEMP_ROOT__ tbody tr:nth-child(even){background:#f8fafc}
#__QTEMP_ROOT__ tbody tr:hover{background:#edf4fb}
#__QTEMP_ROOT__ .note{color:var(--muted);margin:10px 1px 0;font-size:12px}
#__QTEMP_ROOT__ [hidden]{display:none!important}
@media(max-width:650px){
  #__QTEMP_ROOT__ .query{padding:10px}
  #__QTEMP_ROOT__ .tiles{grid-template-columns:repeat(2,minmax(0,1fr));gap:8px}
  #__QTEMP_ROOT__ .tile{padding:10px 12px}
  #__QTEMP_ROOT__ .asset{margin-top:16px}
  #__QTEMP_ROOT__ .plot{height:520px}
  #__QTEMP_ROOT__ .tabs{gap:10px}
  #__QTEMP_ROOT__ .ohlc{gap:4px 9px}
}
@media(max-width:420px){
  #__QTEMP_ROOT__ .controls input[type=search]{flex-basis:100%}
  #__QTEMP_ROOT__ select[data-q=matches]{flex-basis:100%}
  #__QTEMP_ROOT__ .chart-tools button{padding-inline:7px}
}
</style>
<section id="__QTEMP_ROOT__">
  <div class="head"><h1>Qtemp技術指標回測</h1><span class="source muted">Yahoo Finance</span></div>
  <div class="query">
    <div class="controls">
      <input data-q="symbol" type="search" aria-label="股票、ETF 或指數" value="2330" placeholder="2330、0050、AAPL、^TWII">
      <button data-q="search" type="button">搜尋名稱</button>
      <select data-q="matches" aria-label="搜尋結果" hidden></select>
      <button data-q="load" class="primary" type="button">載入並計算</button>
    </div>
  </div>
  <p class="status muted" data-q="status" role="status"></p>
  <div data-q="results" hidden>
    <div class="asset"><h2 data-q="title"></h2><span class="muted" data-q="subtitle"></span></div>
    <div class="tiles">
      <div class="tile">最近收盤<strong data-q="close"></strong></div>
      <div class="tile">日漲跌<strong data-q="change"></strong></div>
      <div class="tile">成交量<strong data-q="volume"></strong></div>
      <div class="tile qtemp">QTemp<strong data-q="score"></strong></div>
    </div>
    <nav class="tabs" aria-label="分析分頁">
      <button data-tab="chart" type="button">行情與 QTemp</button><button data-tab="annual" type="button">年度統計</button>
      <button data-tab="events" type="button">訊號明細</button><button data-tab="returns" type="button">歷史報酬</button>
      <button data-tab="details" type="button">行情與基本資料</button><button data-tab="raw" type="button">日線與公司行動</button><button data-tab="compare" type="button">TradingView 對照</button>
    </nav>
    <div class="panel">
      <div class="filters" data-q="filters" hidden>
        <label>訊號 <select data-q="signal"><option>低估值</option><option>高估值</option><option>底背離</option><option>頂背離</option></select></label>
        <label>近 <select data-q="years" aria-label="統計年數"></select> 年</label>
      </div>
      <div data-q="charttools" hidden>
        <div class="filters chart-tools">
          <div class="range" role="group" aria-label="圖表顯示範圍">
            <button type="button" data-range="22">1 月</button><button type="button" data-range="132">6 月</button>
            <button type="button" data-range="252">1 年</button><button type="button" data-range="all">全部</button>
          </div>
          <button type="button" data-q="reset-chart">重設視圖</button>
        </div>
        <div class="ohlc" data-q="ohlc">
          <span><strong data-value="day"></strong></span>
          <span>開<strong data-value="open"></strong></span><span>高<strong data-value="high"></strong></span>
          <span>低<strong data-value="low"></strong></span><span>收<strong data-value="close"></strong></span>
          <span>量<strong data-value="volume"></strong></span><span>QTemp<strong data-value="score"></strong></span>
        </div>
      </div>
      <div class="filters" data-q="comparetools" hidden><label>TradingView 日 K CSV <input type="file" data-q="csv" accept=".csv,text/csv"></label><button data-q="compare" type="button">逐日比對</button></div>
      <div class="plot" data-q="plot" hidden></div>
      <div data-q="table" hidden></div>
      <p class="note" data-q="note"></p>
    </div>
  </div>
</section>
<script>
(async () => {
  const root = document.getElementById('__QTEMP_ROOT__');
  const q = name => root.querySelector('[data-q="'+name+'"]');
  const tabs = Array.from(root.querySelectorAll('[data-tab]'));
  const num = new Intl.NumberFormat('zh-TW', {maximumFractionDigits:2});
  const pct = x => (x>=0?'+':'')+(x*100).toFixed(2)+'%';
  let data, audit, active = 'chart', chartKey = '', busy = false;
  const status = (value, error=false) => { q('status').textContent=value; q('status').classList.toggle('error',error); };
  const rpc = async (method,args) => {
    const reply = await google.colab.kernel.invokeFunction('__QTEMP_ROOT__.'+method,args,{});
    const result = reply?.data?.['application/json'];
    if (!result) throw new Error('Colab 執行階段未回傳資料，請重新執行程式碼儲存格。');
    if (result.error) throw new Error(result.error);
    return result;
  };
  let chart, observer;
  const ensureChart = () => {
    if (globalThis.LightweightCharts) return Promise.resolve();
    const loadScript=source=>new Promise((resolve,reject)=>{
      const script = document.createElement('script');
      script.src=source;
      script.onload=resolve;
      script.onerror=reject;
      document.head.append(script);
    });
    const bundle='/lightweight-charts@5.1.0/dist/lightweight-charts.standalone.production.js';
    if (!ensureChart.task) ensureChart.task = loadScript('https://cdn.jsdelivr.net/npm'+bundle)
      .catch(()=>loadScript('https://unpkg.com'+bundle)).catch(()=>{
        ensureChart.task=null;
        throw new Error('TradingView 圖表元件載入失敗，請確認網路連線。');
      });
    return ensureChart.task;
  };
  function fitChart(range) {
    if (!chart || !data) return;
    const candles=data.figure.candles;
    if(range==='all') { chart.timeScale().fitContent(); return; }
    const n=candles.length;
    chart.timeScale().setVisibleLogicalRange({from:Math.max(0,n-Number(range)),to:n+4});
  }
  function drawChart(current) {
    const L=globalThis.LightweightCharts, plot=q('plot');
    const up='#bf3445', down='#148161', gold='#986717', axis='#54647a', border='#dce4ed', grid='#edf1f6';
    const crosshair={color:'#8394ab',labelBackgroundColor:'#324d6e'};
    if (observer) observer.disconnect();
    if (chart) chart.remove();
    chart=L.createChart(plot,{
      width:Math.max(1,plot.clientWidth),height:plot.clientHeight,
      localization:{locale:'zh-TW'},
      layout:{background:{color:'#ffffff'},textColor:axis,attributionLogo:true,
              panes:{enableResize:true,separatorColor:border,separatorHoverColor:'#bac9da'}},
      grid:{vertLines:{color:grid},horzLines:{color:grid}},
      crosshair:{mode:L.CrosshairMode.Normal,vertLine:crosshair,horzLine:crosshair},
      rightPriceScale:{borderColor:border},
      handleScroll:{mouseWheel:true,pressedMouseMove:true,horzTouchDrag:true,vertTouchDrag:false},
      handleScale:{mouseWheel:true,axisPressedMouseMove:true,axisDoubleClickReset:true,pinch:true},
      timeScale:{rightOffset:4,borderColor:border,barSpacing:7,minBarSpacing:1}
    });
    const c=chart.addSeries(L.CandlestickSeries,{upColor:up,downColor:down,
      borderVisible:false,wickUpColor:up,wickDownColor:down},0);
    const v=chart.addSeries(L.HistogramSeries,{priceFormat:{type:'volume'},priceLineVisible:false},1);
    const t=chart.addSeries(L.LineSeries,{color:gold,lineWidth:2,priceLineVisible:false,
      lineType:L.LineType.WithSteps,
      autoscaleInfoProvider:()=>({priceRange:{minValue:-105,maxValue:105}})},2);
    chart.panes().forEach((pane,i)=>pane.setStretchFactor([5,1.2,2.2][i]));
    const candles=current.figure.candles, scores=current.figure.scores;
    const byDay=new Map(candles.map((row,i)=>[row[0],[row,scores[i]]]));
    c.setData(candles.map(([time,open,high,low,close])=>({time,open,high,low,close})));
    v.setData(candles.map(([time,open,high,low,close,volume])=>volume===null?{time}:
      {time,value:volume,color:close>=open?'#bf344578':'#14816178'}));
    t.setData(candles.map(([time],i)=>({time,value:scores[i]})));
    [-90,0,90].forEach(price=>t.createPriceLine({price,color:'#aebacf',lineStyle:L.LineStyle.Dotted,axisLabelVisible:false}));
    const markerStyle={buy:['低','#235fbb'],sell:['高','#a73044'],bull:['底','#07745d'],bear:['頂','#925911']};
    L.createSeriesMarkers(c,current.figure.marks.map(([time,type])=>{
      const low=type==='buy'||type==='bull', [text,color]=markerStyle[type];
      return {time,position:low?'belowBar':'aboveBar',shape:low?'arrowUp':'arrowDown',color,text};
    }));
    for(const [start,end,p0,p1,t0,t1,type] of current.figure.segments){
      const color=markerStyle[type][1];
      for(const [pane,y0,y1] of [[0,p0,p1],[2,t0,t1]]){
        const line=chart.addSeries(L.LineSeries,{color,lineWidth:1,lineStyle:L.LineStyle.Dotted,
          lastValueVisible:false,priceLineVisible:false,crosshairMarkerVisible:false,autoscaleInfoProvider:()=>null},pane);
        line.setData([{time:start,value:y0},{time:end,value:y1}]);
      }
    }
    const readouts=Array.from(q('ohlc').querySelectorAll('[data-value]'));
    const update=entry=>{
      if(!entry) return;
      const [[day,o,h,l,close,volume],score]=entry;
      const values={day,open:o,high:h,low:l,close,volume,score};
      for(const node of readouts){
        const value=values[node.dataset.value];
        node.textContent=value===null?'—':typeof value==='number'?num.format(value):value;
      }
    };
    chart.subscribeCrosshairMove(p=>update(byDay.get(p.time)||byDay.get(candles.at(-1)[0])));
    update(byDay.get(candles.at(-1)[0]));
    observer=new ResizeObserver(()=>{if(!plot.hidden&&plot.clientWidth)chart.resize(plot.clientWidth,plot.clientHeight);});
    observer.observe(plot);
    fitChart('252');
    q('reset-chart').onclick=()=>{
      chart.panes().forEach((_,i)=>chart.priceScale('right',i).applyOptions({autoScale:true}));
      fitChart('252');
    };
  }
  function createTable(payload) {
    const holder = document.createElement('div'); holder.className='table';
    if (!payload.rows.length) { holder.textContent='沒有符合條件的資料。'; return holder; }
    const table=document.createElement('table');
    const header=table.createTHead().insertRow();
    for (const name of payload.columns) { const cell=document.createElement('th'); cell.scope='col'; cell.textContent=name; header.append(cell); }
    const body=table.createTBody();
    for (const row of payload.rows) {
      const tr=body.insertRow();
      row.forEach((value,i) => {
        const key=payload.columns[i];
        let text=value;
        if (value==null) text='—';
        else if (typeof value==='boolean') text=value?'是':'否';
        else if (typeof value==='number') text=key.includes('報酬率') ? pct(value) : key==='年度'?String(value):num.format(value);
        else if (/^\d{4}-\d\d-\d\dT00:00:00/.test(value)) text=value.slice(0,10);
        tr.insertCell().textContent=text;
      });
    }
    holder.append(table);
    return holder;
  }
  function render() {
    if (!data) return;
    for (const btn of tabs) btn.setAttribute('aria-pressed',String(btn.dataset.tab===active));
    const chart = active==='chart';
    const plot = q('plot');
    plot.hidden=!chart; q('table').hidden=chart;
    q('filters').hidden=!['annual','events'].includes(active);
    q('charttools').hidden=!chart;
    q('comparetools').hidden=active!=='compare';
    if (chart) {
      q('note').textContent='滾輪縮放、按住拖曳 · 日 K 與訊號採已收盤資料 · '+data.basis;
      if (chartKey===data.symbol) return;
      const current=data;
      ensureChart().then(() => {
        if (data!==current || active!=='chart') return;
        drawChart(current);
        chartKey=current.symbol;
      }).catch(error => status(error.message,true));
      return;
    }
    if (active==='compare') {
      q('table').replaceChildren(...(audit?[createTable(audit.table)]:[]));
      q('note').textContent=audit?audit.note:'匯入同標的的 TradingView 日 K CSV，包含成交量和 QTemp 分數。匯出時請使用相同價格口徑與指標設定。';
      return;
    }
    const years=Number(q('years').value);
    const lowerYear=Number(data.end.slice(0,4))-years+1;
    const signal=q('signal').value;
    let payload, note='';
    if(active==='annual') {
      const source=data.annual[signal];
      payload={columns:source.columns,rows:source.rows.filter(row=>row[0]>=lowerYear)};
      note=data.start+' 至 '+data.end+' · 訊號按確認日歸年 · 當年報酬截至最近收盤';
    } else if(active==='events') {
      payload={columns:data.events.columns,rows:data.events.rows.filter(row=>row[0]===signal && Number(String(row[2]).slice(0,4))>=lowerYear).slice(0,300)};
      note='同類訊號同日列一次，依確認日排序。';
    } else if(active==='returns') { payload=data.returns; note=data.basis+' · 以最近收盤計算'; }
    else if(active==='details') { payload=data.details; note=data.infoMissing?'Yahoo 基本資料暫時無法取得':'Yahoo 報價可能延遲'; }
    else { payload=data.daily; note='最近 120 根還原日 K · 股利與拆併股依 Yahoo 回傳'; }
    q('table').replaceChildren(createTable(payload));
    if(active==='raw') {
      const heading=document.createElement('h3'); heading.textContent='股利與拆併股';
      q('table').append(heading,createTable(data.actions));
    }
    q('note').textContent=note;
  }
  async function load() {
    if (busy) return;
    const symbol=(q('matches').value || q('symbol').value).trim();
    if (!symbol) { status('請輸入標的。',true); return; }
    busy=true; q('load').disabled=true; q('search').disabled=true;
    q('results').hidden=true; status('讀取 Yahoo Finance…');
    try {
      data=await rpc('analyze',[symbol]);
      chartKey=''; audit=null;
      q('title').textContent=data.symbol+' '+(data.name===data.symbol?'':data.name);
      q('subtitle').textContent=[data.market,data.currency,data.end].filter(Boolean).join(' · ');
      q('close').textContent=num.format(data.last);
      q('change').textContent=pct(data.change); q('change').className=data.change>=0?'positive':'negative';
      q('volume').textContent=data.volume==null?'—':num.format(data.volume);
      q('score').textContent=data.qtemp.toFixed(2);
      q('results').hidden=false; status(data.start+' 至 '+data.end+' · '+data.basis);
      render();
    } catch(error) { status(error.message,true); }
    finally { busy=false; q('load').disabled=false; q('search').disabled=false; }
  }
  async function search() {
    if(busy) return;
    const query=q('symbol').value.trim();
    if(!query) return;
    busy=true; q('search').disabled=true; q('load').disabled=true; status('搜尋 Yahoo Finance…');
    try {
      const result=await rpc('search',[query]);
      q('matches').replaceChildren();
      result.choices.forEach(([label,value])=>q('matches').add(new Option(label,value)));
      q('matches').hidden=!result.choices.length;
      status(result.choices.length?'找到 '+result.choices.length+' 個候選標的。':'沒有搜尋結果。');
    } catch(error) { status(error.message,true); }
    finally { busy=false; q('search').disabled=false; q('load').disabled=false; }
  }
  for(let i=1;i<=10;i++) q('years').add(new Option(i,i));
  q('years').value='10';
  for(const button of tabs) button.onclick=()=>{active=button.dataset.tab;render();};
  ['years','signal'].forEach(key=>{q(key).onchange=render;});
  q('symbol').oninput=()=>{q('matches').replaceChildren();q('matches').hidden=true;};
  q('symbol').onkeydown=e=>{if(e.key==='Enter'&&!e.isComposing){e.preventDefault();/[^A-Z0-9.^=_\s-]/i.test(e.currentTarget.value)?search():load();}};
  q('search').onclick=search; q('load').onclick=load;
  root.querySelectorAll('[data-range]').forEach(btn=>{btn.onclick=()=>fitChart(btn.dataset.range);});
  q('compare').onclick=async()=>{
    if (!data||busy) return;
    const file=q('csv').files[0];
    if (!file||file.size>2000000){status('請選取 2 MB 內的 TradingView CSV。',true);return;}
    const current=data;
    busy=true;q('compare').disabled=true;status('比對交易日、收盤價、成交量與 QTemp 分數…');
    try {
      const result=await rpc('compare',[current.symbol,await file.text()]);
      if(data===current){audit=result;status('比對完成。');render();}
    } catch(error) {status(error.message,true);}
    finally {busy=false;q('compare').disabled=false;}
  };
  await load();
})();
</script>
"""

if __name__ == '__main__':
    try:
        display_site()
    except ImportError:
        print('請將整份程式貼進 Google Colab 的一個程式碼儲存格執行。')